# 06 – RQ4: feature matching on HPatches
In this notebook, we compare SIFT, ORB and our small learned descriptor
(TinyDescNet) on the HPatches benchmark. We measure matching accuracy under
viewpoint and illumination changes on the image sequences, and matching
accuracy on the Easy, Hard and Tough patch splits. The same protocol runs on
our own aerial frames in `scripts/matching_experiments.py`.


In [1]:
# Local execution
%matplotlib inline
import sys
from pathlib import Path
PROJ = Path(r'/Users/natalikobrii/Documents/STUDY YORKU & UofT/Computer Vision/Project')
sys.path.insert(0, str(PROJ / 'src')); sys.path.insert(0, str(PROJ))
import torch
print('MPS available:', torch.backends.mps.is_available())
OUT = Path(r'/private/tmp/claude-501/-Users-natalikobrii-Documents-STUDY-YORKU---UofT-Computer-Vision/1ed68044-1713-474c-9d99-5f05afa3a715/scratchpad/nb_out'); OUT.mkdir(parents=True, exist_ok=True)

MPS available: True


In [2]:
# The Okutama sample video
import config
OKUTAMA_VIDEO = PROJ / 'data' / 'raw' / 'okutama' / '1.1.1.mov'
print('Okutama sample:', OKUTAMA_VIDEO, OKUTAMA_VIDEO.exists())

Okutama sample: /Users/natalikobrii/Documents/STUDY YORKU & UofT/Computer Vision/Project/data/raw/okutama/1.1.1.mov True


In [3]:
# The HPatches sequences
config.HPATCHES_DIR = PROJ / 'data' / 'raw' / 'hpatches-sequences-release'
print('HPatches:', config.HPATCHES_DIR,
      len(list(config.HPATCHES_DIR.glob('*'))), 'sequences')

HPatches: /Users/natalikobrii/Documents/STUDY YORKU & UofT/Computer Vision/Project/data/raw/hpatches-sequences-release 116 sequences


In [4]:
# TinyDescNet was trained on aerial Okutama patches (match.train_descriptor,
# scripts/matching_experiments.py). We load the trained weights here so the
# evaluation reproduces the reported numbers deterministically.
tiny_w = PROJ / 'models' / 'tiny_desc.pt'
print('TinyDescNet weights:', tiny_w, tiny_w.exists())

TinyDescNet weights: /Users/natalikobrii/Documents/STUDY YORKU & UofT/Computer Vision/Project/models/tiny_desc.pt True


In [5]:
# Evaluate matching accuracy over all 116 sequences for the three methods.
from eval_match import hpatches_sequences, evaluate_pair, aggregate
from eval_restore import write_csv
from match import build_features
def run(kind, weights=None):
    """Evaluate one method over all HPatches sequence pairs."""
    feats = (build_features(kind, device='cpu', weights=weights)
             if kind == 'tinydesc' else build_features(kind))
    rows = []
    for ref, tgt, H, cond, seq, t in hpatches_sequences(config.HPATCHES_DIR,
                                                        resize_width=1024):
        r = evaluate_pair(feats, ref, tgt, H)
        r.update({'method': kind, 'condition': cond})
        rows.append(r)
    return rows
all_rows = run('sift') + run('orb') + run('tinydesc', weights=str(tiny_w))
agg = aggregate(all_rows)
for r in agg: print(r)
write_csv(agg, OUT/'matching_hpatches.csv')

[W NNPACK.cpp:64] Could not initialize NNPACK! Reason: Unsupported hardware.


Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


{'method': 'orb', 'condition': 'illumination', 'n_kp1': 1992.6491, 'n_kp2': 1986.0246, 'n_covis': 1992.6491, 'n_matches': 410.2491, 'ratio': 0.8, 'ms': 39.9961, 'MMA@1': 0.4344, 'MMA@3': 0.6747, 'MMA@5': 0.7321, 'MMA@10': 0.7595, 'recall@3': 0.1715, 'n_inliers': 342.8772, 'H_corner_err': 46.3785, 'n_pairs': 285}
{'method': 'orb', 'condition': 'viewpoint', 'n_kp1': 1993.4915, 'n_kp2': 1997.6746, 'n_covis': 1884.5153, 'n_matches': 399.6102, 'ratio': 0.8, 'ms': 53.0908, 'MMA@1': 0.2834, 'MMA@3': 0.6696, 'MMA@5': 0.7356, 'MMA@10': 0.7594, 'recall@3': 0.1705, 'n_inliers': 297.3017, 'H_corner_err': 119.8861, 'n_pairs': 295}
{'method': 'sift', 'condition': 'illumination', 'n_kp1': 1591.6316, 'n_kp2': 1540.3439, 'n_covis': 1591.5088, 'n_matches': 380.3614, 'ratio': 0.8, 'ms': 153.2102, 'MMA@1': 0.5216, 'MMA@3': 0.6998, 'MMA@5': 0.7289, 'MMA@10': 0.7471, 'recall@3': 0.2067, 'n_inliers': 308.5719, 'H_corner_err': 31.6119, 'n_pairs': 285}
{'method': 'sift', 'condition': 'viewpoint', 'n_kp1': 1857